# PSY392CPAI Project Analysis
## Vision-Based RL for Flexible Navigation Under Reward and Transition Changes

This notebook analyzes training and evaluation results for three RL agent architectures:
- **PPO** (Proximal Policy Optimization): model-free on-policy baseline
- **SR** (Successor Representation): decouples state features from reward weights for fast revaluation
- **Replay/Dyna**: model-free off-policy DQN with experience replay

**Research question:** Which learning mechanisms enable fast re-planning when rewards or environment dynamics change?

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Add project root to path so we can import src/
project_dir = Path('..').resolve()
sys.path.insert(0, str(project_dir))

results_dir = project_dir / 'results'
csv_dir     = results_dir / 'csv'
fig_dir     = results_dir / 'figures'
fig_dir.mkdir(exist_ok=True)

# Consistent colours
COLORS = {'PPO': '#E15759', 'SR': '#4E79A7', 'Replay': '#59A14F'}
CONDITION_LABELS = {
    'stable':            'Stable',
    'reward_change':     'Reward Change',
    'transition_change': 'Transition Change',
}

print('Results directory:', results_dir)
csv_files = sorted([f.name for f in csv_dir.glob('*.csv')])
print(f'CSV files found ({len(csv_files)}):', csv_files)

## 1. Environment Visualization

The 8×8 GridWorld has three layouts: **Stable** (training layout), **Reward Change** (goal moved to (1,1)), and **Transition Change** (wall configuration shifted). The agent always starts at (6,1).

In [ ]:
from src.envs.gridworld import GridWorldEnv

modes = ['stable', 'reward_change', 'transition_change']

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, mode in zip(axes, modes):
    env = GridWorldEnv(grid_size=8, max_steps=50, change_mode=mode, seed=0)
    env.reset()

    grid = np.ones((8, 8, 3)) * 0.95  # light grey background
    for r, c in env.walls:
        grid[r, c] = [0.2, 0.2, 0.2]   # dark grey walls
    gr, gc = env.goal_pos
    grid[gr, gc] = [0.2, 0.7, 0.2]     # green goal
    ar, ac = env.agent_pos
    grid[ar, ac] = [0.85, 0.15, 0.15]  # red agent

    ax.imshow(grid, origin='upper')
    ax.set_title(CONDITION_LABELS[mode], fontsize=12, fontweight='bold')
    ax.set_xticks(range(8))
    ax.set_yticks(range(8))
    ax.tick_params(labelsize=7)
    ax.grid(True, color='white', linewidth=0.5)
    ax.set_xlabel(f'Goal: {env.goal_pos}  |  Start: {env.agent_pos}', fontsize=8)

legend_elements = [
    mpatches.Patch(facecolor=[0.85, 0.15, 0.15], label='Agent start'),
    mpatches.Patch(facecolor=[0.2, 0.7, 0.2],   label='Goal'),
    mpatches.Patch(facecolor=[0.2, 0.2, 0.2],   label='Wall'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, -0.05))
fig.suptitle('GridWorld Layouts for the Three Evaluation Conditions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(fig_dir / 'env_layouts.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Training Curves

SR and Replay are trained episode-by-episode; PPO uses a frame-based collector.

In [ ]:
def smooth(series, window=15):
    return pd.Series(series).rolling(window, min_periods=1).mean().values

sr_train     = pd.read_csv(csv_dir / 'sr_seed0_train.csv')     if (csv_dir / 'sr_seed0_train.csv').exists()     else None
replay_train = pd.read_csv(csv_dir / 'replay_seed0_train.csv') if (csv_dir / 'replay_seed0_train.csv').exists() else None
ppo_train    = pd.read_csv(csv_dir / 'ppo_seed0_train.csv')    if (csv_dir / 'ppo_seed0_train.csv').exists()    else None

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Episode return
ax = axes[0]
if sr_train is not None:
    ax.plot(sr_train['episode'], smooth(sr_train['episode_return']),
            color=COLORS['SR'], label='SR', linewidth=1.8)
if replay_train is not None:
    ax.plot(replay_train['episode'], smooth(replay_train['episode_return']),
            color=COLORS['Replay'], label='Replay', linewidth=1.8)
if ppo_train is not None:
    ax.plot(ppo_train['collected_frames'] / 1000, smooth(ppo_train['mean_batch_reward'], 5),
            color=COLORS['PPO'], label='PPO (mean batch reward)', linewidth=1.8)
    ax.set_xlabel('Episode  /  Collected Frames (k) for PPO')
else:
    ax.set_xlabel('Episode')
ax.axhline(0.9, color='grey', linestyle='--', linewidth=0.8, alpha=0.6, label='~Perfect (0.9)')
ax.set_ylabel('Episode Return (smoothed w=15)')
ax.set_title('Training Return')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Training loss
ax = axes[1]
if sr_train is not None:
    q95 = sr_train['total_loss'].quantile(0.95)
    ax.plot(sr_train['episode'], smooth(sr_train['total_loss'].clip(upper=q95)),
            color=COLORS['SR'], label='SR total loss (clipped 95th %ile)', linewidth=1.8)
if replay_train is not None:
    ax.plot(replay_train['episode'], smooth(replay_train['q_loss']),
            color=COLORS['Replay'], label='Replay Q-loss', linewidth=1.8)
if ppo_train is not None:
    ax.plot(ppo_train['collected_frames'] / 1000, smooth(ppo_train['total_loss'], 5),
            color=COLORS['PPO'], label='PPO total loss', linewidth=1.8)
ax.set_xlabel('Episode  /  Collected Frames (k) for PPO')
ax.set_ylabel('Loss (smoothed)')
ax.set_title('Training Loss')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

fig.suptitle('Training Curves: All Three Agents', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(fig_dir / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Per-Agent Evaluation Across Conditions

Agents were evaluated every 25 episodes (SR/Replay) or every 10 batches (PPO) on 20 greedy episodes each.

In [ ]:
def load_eval(prefix):
    frames = []
    for cond in ['stable', 'reward_change', 'transition_change']:
        path = csv_dir / f'{prefix}_eval_{cond}.csv'
        if path.exists():
            frames.append(pd.read_csv(path))
    return pd.concat(frames, ignore_index=True) if frames else None

ppo_eval    = load_eval('ppo_seed0')
sr_eval     = load_eval('sr_seed0')
replay_eval = load_eval('replay_seed0')

cond_colors = {'stable': '#4E79A7', 'reward_change': '#E15759', 'transition_change': '#59A14F'}

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (name, df) in zip(axes, [('PPO', ppo_eval), ('SR', sr_eval), ('Replay', replay_eval)]):
    if df is None:
        ax.text(0.5, 0.5, 'No data yet', ha='center', va='center', transform=ax.transAxes, fontsize=12)
        ax.set_title(name)
        continue
    x_col = 'episode' if 'episode' in df.columns else 'batch_idx'
    for cond, color in cond_colors.items():
        sub = df[df['condition'] == cond]
        if not sub.empty:
            ax.plot(sub[x_col], sub['success_rate'], color=color,
                    label=CONDITION_LABELS[cond], marker='o', markersize=4, linewidth=1.8)
    ax.set_title(name, fontsize=12, fontweight='bold', color=COLORS[name])
    ax.set_xlabel('Episode' if x_col == 'episode' else 'Batch')
    ax.set_ylim(-0.05, 1.15)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.grid(True, alpha=0.3)
    if ax is axes[0]:
        ax.set_ylabel('Success Rate')
    ax.legend(fontsize=8)

fig.suptitle('Evaluation Success Rate by Agent and Condition', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(fig_dir / 'eval_per_agent.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Cross-Agent Comparison

Key figure for testing the dissociation hypothesis: success rates averaged over the **last 3 evaluation checkpoints** per agent per condition.

In [ ]:
conditions = ['stable', 'reward_change', 'transition_change']
agent_eval = {'PPO': ppo_eval, 'SR': sr_eval, 'Replay': replay_eval}

def final_mean(df, cond, last_n=3):
    if df is None:
        return np.nan
    sub = df[df['condition'] == cond]
    return sub['success_rate'].tail(last_n).mean() if not sub.empty else np.nan

x = np.arange(len(conditions))
width = 0.25
offsets = [-width, 0, width]

fig, ax = plt.subplots(figsize=(9, 5))
for i, (agent, df) in enumerate(agent_eval.items()):
    vals = [final_mean(df, c) for c in conditions]
    bars = ax.bar(x + offsets[i], vals, width=width, label=agent,
                  color=COLORS[agent], alpha=0.88, edgecolor='white')
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                    f'{v:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels([CONDITION_LABELS[c] for c in conditions], fontsize=11)
ax.set_ylabel('Mean Success Rate (last 3 checkpoints)', fontsize=11)
ax.set_ylim(0, 1.3)
ax.set_title('Cross-Agent Comparison: Adaptation to Environmental Changes',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=11)
ax.axhline(1.0, color='grey', linestyle='--', linewidth=0.8, alpha=0.5)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(fig_dir / 'cross_agent_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. SR Reward Weights Evolution

Unique to the SR agent: the reward weights **w** should grow during training as the agent learns what features predict reward.

In [ ]:
if sr_train is not None and 'reward_weights_norm' in sr_train.columns:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(sr_train['episode'], sr_train['reward_weights_norm'],
                 color=COLORS['SR'], linewidth=1.8)
    axes[0].set_xlabel('Episode')
    axes[0].set_ylabel('L2 norm of w')
    axes[0].set_title('Reward Weight Norm')
    axes[0].grid(True, alpha=0.3)

    q95 = sr_train['total_loss'].quantile(0.95)
    axes[1].plot(sr_train['episode'],
                 smooth(sr_train['total_loss'].clip(upper=q95), 20),
                 color=COLORS['SR'], linewidth=1.8)
    axes[1].set_xlabel('Episode')
    axes[1].set_ylabel('Total Loss (clipped)')
    axes[1].set_title('SR Total Loss (95th-percentile clip)')
    axes[1].grid(True, alpha=0.3)

    fig.suptitle('SR Training Diagnostics', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(fig_dir / 'sr_diagnostics.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('SR training data not yet available — run train_sr.py first.')

## 6. Summary Statistics Table

In [ ]:
rows = []
for agent, df in agent_eval.items():
    for cond in conditions:
        if df is not None:
            sub = df[df['condition'] == cond]
            if not sub.empty:
                tail = sub.tail(3)
                rows.append(dict(
                    Agent=agent,
                    Condition=CONDITION_LABELS[cond],
                    SuccessRate_Mean=f"{tail['success_rate'].mean():.3f}",
                    SuccessRate_Std=f"{tail['success_rate'].std():.3f}",
                    AvgReturn=f"{tail['avg_return'].mean():.3f}",
                    AvgSteps=f"{tail['avg_steps'].mean():.1f}",
                ))
            else:
                rows.append(dict(Agent=agent, Condition=CONDITION_LABELS[cond],
                                 SuccessRate_Mean='N/A', SuccessRate_Std='N/A',
                                 AvgReturn='N/A', AvgSteps='N/A'))
        else:
            rows.append(dict(Agent=agent, Condition=CONDITION_LABELS[cond],
                             SuccessRate_Mean='No data', SuccessRate_Std='-',
                             AvgReturn='-', AvgSteps='-'))

summary_df = pd.DataFrame(rows)
display(summary_df) if 'display' in dir() else print(summary_df.to_string(index=False))

## 7. Key Findings

### Hypotheses

| ID | Prediction | Test |
|---|---|---|
| **H1** | SR adapts faster to reward changes than PPO | SR success_rate(reward_change) > PPO success_rate(reward_change) |
| **H2** | Replay adapts better to transition changes than SR | Replay success_rate(transition_change) ≥ SR success_rate(transition_change) |
| **H3** | Dissociation/crossover across conditions | SR best on reward_change; Replay best on transition_change |

### Mechanistic Interpretation

**Successor Representation (SR):**  
The SR factorises the value function as Q(s,a) = ⟨ψ(s,a), w⟩ where ψ captures expected future state occupancies and **w** are reward weights. After a reward change (new goal location), only **w** needs updating — the learned ψ transfer across reward functions. This is the key prediction of the SR framework (Dayan 1993; Russek et al. 2017): reward revaluation should be fast because the occupancy model is reward-agnostic.

**Replay/Dyna:**  
The replay buffer stores raw (s,a,r,s') transitions. When transition dynamics change (walls shift), replaying stored experience from the *new* layout provides training signal about the updated graph structure without additional environment interaction. This is conceptually similar to offline planning in Dyna-Q (Sutton 1990), but without an explicit world model.

**PPO baseline:**  
On-policy PPO has no mechanism for rapid transfer — it must re-explore after any perturbation. It serves as the lower-bound reference.

### Cognitive Science Connection

This dissociation mirrors the hippocampal-striatal debate (Daw et al. 2005): the hippocampus is proposed to maintain SR-like representations supporting rapid reward revaluation (Momennejad et al. 2017), while the striatum stores cached model-free values that require replay during sleep/rest to update. A computational model that shows the SR advantage on reward changes and a replay advantage on transition changes provides a mechanistic account of why both systems are adaptive under different types of environmental perturbation.